# 02 — OCR Evaluation with CER and WER

**Question:** How can we compare OCR systems in a reproducible way instead of saying that one output looks better?

We will implement the metric ourselves, test it on a known example, OCR one page with three configurations, and save a comparison table.

## 1. Setup and data

Ground truth is a human-checked transcription. In a real evaluation it should be created independently and frozen before model tuning.

In [ ]:
from pathlib import Path
import csv
import re
import shutil
import cv2
import matplotlib.pyplot as plt
import pytesseract

HERE = Path.cwd()
if not (HERE / 'data').exists():
    HERE = HERE / '02_ocr_evaluation'
DATA, OUTPUT = HERE / 'data', HERE / 'output'
OUTPUT.mkdir(exist_ok=True)
reference = (DATA / 'ground_truth.txt').read_text(encoding='utf-8')
image = cv2.imread(str(DATA / 'imageTextN.png'))
print('Reference characters:', len(reference), 'words:', len(reference.split()))
plt.figure(figsize=(14, 6)); plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB)); plt.axis('off');

## 2. Levenshtein dynamic programming

The matrix stores the cheapest edit cost for every reference/hypothesis prefix pair. A traceback converts the final distance into substitutions (S), deletions (D), and insertions (I). We prefer substitutions when several paths have equal cost so the breakdown is deterministic.

In [ ]:
def edit_counts(reference_tokens, hypothesis_tokens):
    rows, cols = len(reference_tokens) + 1, len(hypothesis_tokens) + 1
    cost = [[0] * cols for _ in range(rows)]
    operation = [[''] * cols for _ in range(rows)]
    for i in range(1, rows):
        cost[i][0], operation[i][0] = i, 'D'
    for j in range(1, cols):
        cost[0][j], operation[0][j] = j, 'I'
    for i in range(1, rows):
        for j in range(1, cols):
            if reference_tokens[i - 1] == hypothesis_tokens[j - 1]:
                cost[i][j], operation[i][j] = cost[i - 1][j - 1], 'M'
            else:
                choices = [(cost[i-1][j-1] + 1, 'S'),
                           (cost[i-1][j] + 1, 'D'),
                           (cost[i][j-1] + 1, 'I')]
                cost[i][j], operation[i][j] = min(choices, key=lambda item: item[0])
    counts = {'S': 0, 'D': 0, 'I': 0}
    i, j = len(reference_tokens), len(hypothesis_tokens)
    while i or j:
        op = operation[i][j]
        if op in ('M', 'S'):
            if op == 'S': counts['S'] += 1
            i, j = i - 1, j - 1
        elif op == 'D':
            counts['D'] += 1; i -= 1
        else:
            counts['I'] += 1; j -= 1
    return counts

def normalize(text):
    return re.sub(r'\s+', ' ', text.strip().lower())

def error_rate(reference_text, hypothesis_text, level='character'):
    ref, hyp = normalize(reference_text), normalize(hypothesis_text)
    ref_tokens = list(ref) if level == 'character' else ref.split()
    hyp_tokens = list(hyp) if level == 'character' else hyp.split()
    counts = edit_counts(ref_tokens, hyp_tokens)
    errors = sum(counts.values())
    rate = errors / len(ref_tokens) if ref_tokens else float(errors > 0)
    return rate, counts, len(ref_tokens)

## 3. Sanity-check the implementation

`kitten → sitting` has edit distance 3: two substitutions and one insertion. A known example catches mistakes before we trust the metric on OCR output.

In [ ]:
check = edit_counts(list('kitten'), list('sitting'))
print(check, 'total distance =', sum(check.values()))
assert sum(check.values()) == 3
assert error_rate('Exact match', 'exact   match')[0] == 0

## 4. Create three OCR inputs

The recognizer and page segmentation mode stay fixed. Only preprocessing changes, so this is an ablation of the image preparation choice.

In [ ]:
gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
denoised = cv2.bilateralFilter(gray, 7, 35, 35)
otsu = cv2.threshold(denoised, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)[1]
adaptive = cv2.adaptiveThreshold(denoised, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                 cv2.THRESH_BINARY, 31, 15)
candidates = {'raw': image, 'otsu': otsu, 'adaptive': adaptive}
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for axis, (name, candidate) in zip(axes, candidates.items()):
    shown = cv2.cvtColor(candidate, cv2.COLOR_BGR2RGB) if candidate.ndim == 3 else candidate
    axis.imshow(shown, cmap='gray'); axis.set_title(name); axis.axis('off')
plt.tight_layout();

## 5. OCR and evaluate

CER uses character tokens; WER uses word tokens. Both are normalized by the reference length, not the hypothesis length. The CSV stores operation counts so the final number remains auditable.

In [ ]:
if shutil.which('tesseract') is None:
    raise RuntimeError('Install Tesseract first: brew install tesseract')
rows = []
for name, candidate in candidates.items():
    prediction = pytesseract.image_to_string(candidate, config='--psm 6').strip()
    (OUTPUT / f'{name}_ocr.txt').write_text(prediction, encoding='utf-8')
    cer, char_ops, char_count = error_rate(reference, prediction, 'character')
    wer, word_ops, word_count = error_rate(reference, prediction, 'word')
    rows.append({'method': name, 'cer': cer, 'wer': wer,
                 'char_S': char_ops['S'], 'char_D': char_ops['D'], 'char_I': char_ops['I'],
                 'reference_characters': char_count, 'reference_words': word_count})

with (OUTPUT / 'metrics.csv').open('w', newline='', encoding='utf-8') as handle:
    writer = csv.DictWriter(handle, fieldnames=rows[0].keys())
    writer.writeheader(); writer.writerows(rows)

print(f"{'method':<10} {'CER':>8} {'WER':>8} {'S':>5} {'D':>5} {'I':>5}")
for row in rows:
    print(f"{row['method']:<10} {row['cer']:>8.3f} {row['wer']:>8.3f} "
          f"{row['char_S']:>5} {row['char_D']:>5} {row['char_I']:>5}")

## Reflection

1. Which method has the lowest CER? Does it also have the lowest WER?
2. Which edit operation dominates, and what visual failure could cause it?
3. Would case and whitespace normalization be appropriate for a legal transcription task?
4. This is only one page. What claims are not justified by this result?